<a href="https://colab.research.google.com/github/minju0236/-gyeonggi-do_map/blob/main/Day2(260224)_cloudflare%26node%26html%EC%97%B0%EA%B2%B0_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

node js 사용을 위한 npm 초기화 및 express 설치

In [ ]:
!npm init -y
!npm install express

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}



⠙⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 65 packages, and audited 66 packages in 6s
⠼
⠼22 packages are looking for funding
⠼  run `npm fund` for details
⠼
found 0 vulnerabilities
⠼

was 작성 및 실행(파일 생성 및 정의)

In [ ]:
%%writefile server.js

/*
  Express 웹 서버를 사용하여
  1) HTML 파일을 브라우저에 내려주고
  2) 폼 데이터를 받아
  3) 템플릿 HTML 파일을 읽어 값 치환 후
  4) 새로운 HTML 파일을 생성하여 응답하는 구조
*/

const express = require("express");   // 웹 서버 프레임워크
const path = require("path");         // 파일 경로 처리를 위한 Node 기본 모듈
const fs = require("fs");             // 파일 읽기/쓰기(File System) 모듈

const app = express();                // Express 애플리케이션 객체 생성

/*
  express.urlencoded 미들웨어
  ----------------------------------------------------
  HTML <form>이 기본적으로 보내는 데이터 형식은
  application/x-www-form-urlencoded 이다.

  이 미들웨어는 POST 요청으로 전달된 form 데이터를
  req.body 객체로 파싱해준다.

  예:
    customer=홍길동&item=커피&qty=2
  → req.body = { customer: "홍길동", item: "커피", qty: "2" }
*/
app.use(express.urlencoded({ extended: true }));

/*
  ----------------------------------------------------
  GET "/"
  ----------------------------------------------------
  브라우저가 http://localhost:3000 에 접속하면 실행된다.

  res.sendFile()은 지정한 HTML 파일을
  그대로 브라우저에 전달한다.

  즉, 서버가 HTML 파일을 렌더링하는 것이 아니라
  "파일을 전달"하면 브라우저가 렌더링한다.
*/
app.get("/", (req, res) => {
  res.sendFile(path.join(__dirname, "index.html"));
});

/*
  ----------------------------------------------------
  POST "/submit"
  ----------------------------------------------------
  사용자가 폼을 제출하면 실행된다.

  처리 흐름:
    1) 사용자가 입력한 값(req.body) 추출
    2) 주문서 템플릿 HTML 파일 읽기
    3) 템플릿의 {{변수}}를 실제 값으로 치환
    4) 완성된 HTML을 order.html 파일로 저장
    5) 생성된 HTML 파일을 브라우저에 응답
*/
app.post("/submit", (req, res) => {

  // 구조분해 할당으로 form 데이터 추출
  // req.body는 express.urlencoded 미들웨어가 만들어줌
  const { customer, item, qty } = req.body;

  /*
    order_template.html 파일을 읽는다.
    utf8로 읽어야 문자열(String)로 처리 가능하다.
  */
  const orderTemplateContent = fs.readFileSync(
    path.join(__dirname, "order_template.html"),
    "utf8"
  );

  /*
    템플릿 치환 작업
    ---------------------------------
    예:
      <p>주문자: {{customer}}</p>

    replaceAll을 이용해
    {{customer}} → 실제 입력값으로 교체
  */
  const completedOrderHtml = orderTemplateContent
    .replaceAll("{{customer}}", customer)
    .replaceAll("{{item}}", item)
    .replaceAll("{{qty}}", qty);

  /*
    완성된 HTML을 새로운 파일(order.html)로 저장한다.
    → 이 시점에 서버의 파일 시스템에 실제 파일이 생성됨
  */
  fs.writeFileSync(
    path.join(__dirname, "order.html"),
    completedOrderHtml
  );

  /*
    생성된 order.html 파일을 브라우저에 전송한다.
    응답은 문자열이 아니라 "HTML 파일"이다.
  */
  res.sendFile(path.join(__dirname, "order.html"));
});

/*
  ----------------------------------------------------
  서버 실행
  ----------------------------------------------------
  포트 3000에서 HTTP 서버를 시작한다.

  브라우저 접속 주소:
    http://localhost:3000
*/
app.listen(3000, () => {
  console.log("Server started at http://localhost:3000");
});

Writing server.js


In [ ]:
%%writefile index.html

<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <title>주문 입력</title>
</head>
<body>
  <h1>주문 입력 화면</h1>

  <form method="POST" action="/submit">
    <p>
      주문자:
      <input name="customer" required>
    </p>

    <p>
      상품명:
      <input name="item" required>
    </p>

    <p>
      수량:
      <input type="number" name="qty" min="0" value="0" required>
    </p>

    <button type="submit">주문 접수</button>
  </form>
</body>
</html>

Writing index.html


In [ ]:
%%writefile order_template.html

<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <title>주문서</title>
</head>
<body>
  <h1>주문서</h1>

  <p>주문자: {{customer}}</p>
  <p>상품명: {{item}}</p>
  <p>수량: {{qty}}</p>

  <hr>
  <a href="/">새 주문하기</a>
</body>
</html>

Writing order_template.html


서버 실행

In [ ]:
!nohup node server.js > server.log 2>&1 &

In [ ]:
!tail -n 20 server.log

Server started at http://localhost:3000


cloudflare 설치 후 was와 ws 3000포트로 연결

In [ ]:
!npm install -g cloudflared

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 1 package in 2s
⠧

In [ ]:
!nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &

In [ ]:
!tail -n 20 tunnel.log

2026-02-24T02:40:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-24T02:40:03Z INF Requesting new quick Tunnel on trycloudflare.com...


ws로 url 생성

In [ ]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log |tail -n 1

https://epson-camps-societies-encoding.trycloudflare.com
